# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
import sys
sys.path.append("..")

from common_config import llm_connect
model = "gpt-5.4"

temperatuyre = 0
max_tokens = 2086

llm = llm_connect(model = model)

True

## 1. PromptTemplate - 단일 문자열 템플릿 생성법

In [2]:
from langchain_core.prompts import PromptTemplate

# Template 객체 설정. 사용자의 User prompt(질문)를 받음.
template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘") # {topic} : template 변수. f 넣지 않는다.

# Prompt 생성 자동화. 변수 채우기
prompt = template.invoke({"topic": "RAG"}) # LLM과 통신 X. 객체를 invoke. dictionary 형태로 넣음. 위에서 정의한 topic 칸에 RAG를 넣는다.
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿 생성법

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([ # List로 넣는다.
    ("system", "당신은 {role} 전문가입니다."),      # System Prompt
    ("human", "{question}")                      # 사용자의 질문.
])

messages = chat_prompt.invoke({                 # Dictionary로 넣는다.
    "role": "파이썬",                            # 위의 role 자리에 파이썬이 들어감.
    "question": "데코레이터가 뭔가요?"             # 위의 question 자리에 "데코레이터가 뭔가요?" 가 들어감.
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [5]:
from langchain_openai import ChatOpenAI

import sys
sys.path.append("..")

from common_config import llm_connect
model = "gpt-5.4"

temperatuyre = 0
max_tokens = 2086

llm = llm_connect(model = model)

chat_prompt = ChatPromptTemplate.from_messages([ # List로 넣는다.
    ("system", "당신은 {role} 전문가입니다."),      # System Prompt
    ("human", "{question}")                      # 사용자의 질문.
])

messages = chat_prompt.invoke({  # Dictionary로 넣는다.
    "role": "머신러닝",           # 위의 role 자리에 머신러닝이 들어감.
    "question": "과적합이 뭔가요?" # 위의 question 자리에 "과적합이 뭔가요?" 가 들어감.
})

response = llm.invoke(messages) # 이제는 LLM에 messages를 넣는다.
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에 너무 딱 맞게 배워서**, 새로운 데이터에서는 성능이 떨어지는 현상입니다.

쉽게 말하면:

- **학습 데이터 문제는 잘 푼다**
- 그런데 **시험 데이터는 못 푼다**

즉, 데이터의 **진짜 패턴**이 아니라 **우연한 잡음이나 세부사항까지 외워버린 상태**입니다.

### 예시
학생이 개념을 이해한 게 아니라 **기출문제를 통째로 암기한 경우**와 비슷합니다.  
기출문제는 잘 풀지만, 조금만 다르게 나오면 못 풉니다.

### 왜 생기나요?
보통 이런 경우에 과적합이 잘 생깁니다:
- 모델이 너무 복잡할 때
- 학습 데이터가 너무 적을 때
- 학습을 너무 오래 할 때
- 데이터에 잡음이 많을 때

### 어떻게 알 수 있나요?
보통:
- **훈련 성능은 매우 좋음**
- **검증/테스트 성능은 나쁨**

이런 패턴이 보이면 과적합을 의심합니다.

### 방지 방법
- 데이터를 더 많이 모으기
- 모델을 단순하게 만들기
- 정규화(regularization) 사용
- 드롭아웃(dropout) 사용
- 조기 종료(early stopping)
- 데이터 증강(data augmentation)

원하시면 제가 **과적합 / 과소적합 / 일반화** 차이도 같이 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

- 이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.
- LLM은 과거 대화 내용 기억하지 못하고 맥락을 이어가는 질문을 받았을 때 답 하기 위해 필요함.

In [6]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),  # System prompt는 한 번만 넣으면 됨. 이후 대화에선 그닥 필요 X.
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록. 백엔드가 가지고 있다가 LLM에게 넘겨야 함.
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({     # 이전 대화 내용 넘김. Dictionary 형태.
    "history": history,             # chat prompt의 history 자리에 넣음.
    "question": "내 이름이 뭐라고?"   # char pormpt의 question 자리에 넣음.
})

response = llm.invoke(messages)    # history 내용 포함한 messages를 llm에 넣어 답을 받음.
print(response.content)

홍길동님이세요.


In [7]:
messages

# 대화의 history가 누적되었음을 확인 가능(System prompt -> Human Message -> AI Message -> Human Message).
# 대화 session id 만들어서 sql로 구성하여 저장한다. 이후 답변 요청 오면 db에서 query 하여 history에 집어넣음.

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={})])

In [8]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [10]:
template = ChatPromptTemplate.from_messages([ # system prompt엔 톤 앤 매너 등 일관되게 유지되어야 할 부분을 넣어야 함. 사용자는 실제 문장만 넣으면 의도되로 돌아가게.
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:                                         
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),                                         #  """ 를 써서 문자열 만들면 간격 등 반영한 결과 만들어짐.
     # One shot으로 예시 줌.
    ("human", "다음 문장을 번역해줘: {text}") # 프런트엔드에서 입력 받은 text를 여기에 넣는다.
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

한국어: 오늘 날씨가 정말 좋네요.  
영어: The weather is really nice today.


In [11]:
messages = template.invoke({
    "sorted_language": "영어",
    "target_language": "한국어",
    "text": "Don't go gentle into the dark night."
})

response = llm.invoke(messages)
print(response.content)

영어: Don't go gentle into the dark night.  
한국어: 어두운 밤 속으로 순순히 들어가지 마세요.
